## System Architecture Patterns

### Lambda Architecture (Batch + Real-time)
```
                    ┌─────────────┐
                    │  Raw Data   │
                    └─────┬───────┘
                          │
                ┌─────────┴─────────┐
                │                   │
        ┌───────▼──────┐    ┌──────▼────────┐
        │ Batch Layer  │    │ Speed Layer    │
        │ (Historical) │    │ (Real-time)    │
        └───────┬──────┘    └──────┬────────┘
                │                   │
        ┌───────▼───────────────────▼──────┐
        │   Serving Layer (Merge results)   │
        └─────────────────────────────────┘
```

**Use case**: Financial predictions, fraud detection

### Kappa Architecture (Event streaming)
```
Data Stream → Stream Processing → Model Inference → Results
```

**Use case**: Real-time recommendations, anomaly detection

### Microservices Architecture
```
┌─────────────────────────────────────────┐
│         API Gateway                     │
└─┬──────────────┬──────────────┬────────┘
  │              │              │
┌─▼──┐  ┌──────▼─────┐  ┌─────▼──┐
│ ML │  │ ML Service │  │ ML     │
│ 1  │  │ 2          │  │ Service│
│    │  │            │  │ 3      │
└────┘  └────────────┘  └────────┘
```

## Data Pipeline Design

### Pipeline Stages
1. **Data Ingestion**
   - Sources: APIs, databases, logs, streams
   - Frequency: batch, real-time, hybrid
   - Error handling and retries

2. **Data Transformation**
   - Cleaning: missing values, outliers
   - Normalization and standardization
   - Feature engineering
   - Data validation

3. **Feature Storage**
   - Feature warehouse/store
   - Reusable features
   - Point-in-time correctness

4. **Model Training**
   - Train on historical data
   - Experiment tracking
   - Validation and testing

### Technology Stack
| Component | Options |
|-----------|----------|
| Ingestion | Kafka, AWS Kinesis, GCP Pub/Sub |
| Processing | Spark, Flink, DataFlow |
| Storage | Parquet, DeltaLake, Iceberg |
| Feature Store | Tecton, Feast, offline |
| Orchestration | Airflow, Prefect, Dagster |

## Model Serving Architecture

### Serving Patterns
1. **Batch Serving**
   - Compute predictions offline
   - Store predictions in database
   - Query at request time
   - Pro: Simple, scalable
   - Con: Stale predictions

2. **Real-time Serving**
   - Compute on request
   - Requires low latency
   - Pro: Fresh predictions
   - Con: Infrastructure intensive

3. **Streaming Serving**
   - Continuous prediction streams
   - For fraud/anomaly detection
   - State management needed

### Optimization Strategies
```
Request → Feature Lookup → Cache Hit? → Model Inference → Response
                          └─ Yes → Return cached
```

- **Caching**: Recent features, predictions
- **Model Compression**: Quantization, pruning, distillation
- **Batch Processing**: Group requests
- **GPU/TPU**: For large models
- **ONNX Runtime**: Model format optimization

## Scalability Considerations

### Data Scaling
- **Volume**: Terabytes to petabytes
- **Velocity**: Streaming vs. batch
- **Variety**: Different data formats
- Solution: Distributed processing (Spark, Hadoop)

### Model Scaling
- **Size**: Billions of parameters
- **Latency**: <100ms requirement typical
- **Throughput**: Millions of predictions/day
- Solution: Model sharding, ensemble serving

### Inference Scaling
```
Load Balancer
     │
     ├─→ Inference Pod 1
     ├─→ Inference Pod 2
     ├─→ Inference Pod 3
     └─→ Inference Pod N
```

- Kubernetes for orchestration
- Horizontal scaling: More instances
- Vertical scaling: Bigger instances
- Auto-scaling based on load

### Cost Optimization
- Reserved capacity vs. on-demand
- Spot instances for batch jobs
- Model quantization reduces memory
- Caching reduces inference calls

## Monitoring and Observability

### Metrics to Monitor
1. **System Metrics**
   - Latency (p50, p95, p99)
   - Throughput (req/sec)
   - Error rate
   - Resource usage (CPU, GPU, memory)

2. **ML Metrics**
   - Model accuracy
   - Data drift detection
   - Model performance by segment
   - Prediction distribution shifts

3. **Business Metrics**
   - Model impact on business KPIs
   - ROI tracking
   - User satisfaction

### Data Drift Detection
```python
# Compare current data distribution to training data
def detect_drift(current_data, baseline_data):
    for feature in features:
        ks_stat = ks_test(current_data[feature], 
                         baseline_data[feature])
        if ks_stat > threshold:
            alert(f"Drift detected in {feature}")
```

### Alerting
- Accuracy drop > 5%
- Latency > 200ms (p95)
- Error rate > 1%
- Data drift detected
- Prediction distribution change

## Deployment Strategies

### Canary Deployment
```
Old Model: 99% traffic
New Model: 1% traffic (canary)
         ↓ (monitor 1 hour)
Old Model: 90%
New Model: 10%
         ↓ (monitor)
New Model: 100%
```

### A/B Testing
```
Group A: Old Model (50%)
Group B: New Model (50%)
Measure: Business metrics
```

### Blue-Green Deployment
```
Blue (current)  ←─ Load Balancer
Green (new)   ↙

Test Green, switch traffic, keep Blue as fallback
```

### Rollback Plan
- Always maintain previous model version
- One-click rollback capability
- Clear triggers for rollback
- Post-incident review

## ML Architecture Best Practices

### Design Principles
1. **Separation of Concerns**
   - Data pipeline separate from model training
   - Training separate from serving
   - Feature computation reused

2. **Reproducibility**
   - Version all inputs: data, code, parameters
   - Reproduce training results
   - Track experiments

3. **Versioning**
   - Model versioning and tracking
   - Feature store versioning
   - Data versioning

4. **Testing**
   - Unit tests for data pipelines
   - Integration tests
   - Model validation tests
   - Regression tests

### Architectural Checklist
- [ ] Clear separation of concerns
- [ ] Scalable for expected volume
- [ ] Fault tolerance built in
- [ ] Monitoring and alerting
- [ ] Versioning and reproducibility
- [ ] Security and privacy
- [ ] Documentation and runbooks
- [ ] Team can operate it